# Feature Extraction
Derive new features, encode categoricals, and produce the final feature matrix.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_preprocessed.csv', index_col='ID')
df['cpu_ghz'] = df['cpu_ghz'].fillna(df['cpu_ghz'].median())
df.shape

(1303, 20)

## Derived Features

In [2]:
df['ppi'] = np.sqrt(df['res_w']**2 + df['res_h']**2) / df['Inches']
df['ppi'] = df['ppi'].replace([np.inf, -np.inf], np.nan).fillna(df['ppi'].median())

df['total_storage_gb'] = df['ssd_gb'] + df['hdd_gb'] + df['flash_gb']
df['has_ssd']          = (df['ssd_gb'] > 0).astype(int)
df['has_hdd']          = (df['hdd_gb'] > 0).astype(int)
df['has_dedicated_gpu'] = (df['gpu_tier'] > 0).astype(int)

# Ordinal CPU tier — captures the performance ordering
cpu_tier_ord = {'Atom': 0, 'Celeron': 1, 'Pentium': 2, 'Core_m': 3,
                'i3': 4, 'i5': 5, 'i7': 6, 'i9': 7, 'Xeon': 7, 'Ryzen': 5, 'Other': 3}
df['cpu_tier_score'] = df['cpu_tier'].map(cpu_tier_ord).fillna(3)

# Log2 RAM: 4→2, 8→3, 16→4, 32→5, 64→6 — more linear relationship with price
df['ram_log2'] = np.log2(df['ram_gb'].clip(lower=1))

df['log_price'] = np.log1p(df['Price'])

## Encoding

In [3]:
cat_cols = ['Company', 'TypeName', 'cpu_brand', 'cpu_tier', 'gpu_brand', 'os']
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)

## Drop Redundant Raw Columns

In [4]:
drop_cols = ['res_w', 'res_h']
df_enc = df_enc.drop(columns=drop_cols)
print(df_enc.shape)

(1303, 65)


## Correlation with Price

In [5]:
corr = df_enc.corr()['Price'].abs().sort_values(ascending=False)
print(corr.head(25))

Price                   1.000000
log_price               0.927584
ram_gb                  0.743007
ram_log2                0.742176
ssd_gb                  0.670682
cpu_tier_score          0.594339
cpu_tier_i7             0.556782
TypeName_Notebook       0.549248
has_ssd                 0.513609
ppi                     0.473487
gpu_tier                0.409141
cpu_ghz                 0.396029
TypeName_Gaming         0.375789
gpu_brand_Nvidia        0.348797
cpu_tier_Celeron        0.309804
cpu_tier_i3             0.284372
cpu_gen                 0.276667
TypeName_Ultrabook      0.255658
ips_panel               0.252208
TypeName_Workstation    0.249752
Company_Razer           0.233756
weight_kg               0.210370
Company_Acer            0.208349
os_Linux                0.203776
gpu_brand_AMD           0.199415
Name: Price, dtype: float64


## Save

In [6]:
df_enc.to_csv('data_features.csv')
df_enc.head()

,Inches,Price,ram_gb,weight_kg,touchscreen,ips_panel,cpu_ghz,cpu_gen,ssd_gb,hdd_gb,...,cpu_tier_i5,cpu_tier_i7,gpu_brand_AMD,gpu_brand_Intel,gpu_brand_Nvidia,os_Linux,os_NoOS,os_Win10,os_WinOther,os_macOS
ID,,,,,,,,,,,,,,,,,,,,,
0,13.3,71378.6832,8,1.37,0,1,2.3,0,128,0,...,True,False,False,True,False,False,False,False,False,True
1,13.3,47895.5232,8,1.34,0,0,1.8,0,0,0,...,True,False,False,True,False,False,False,False,False,True
2,15.6,30636.0000,8,1.86,0,0,2.5,7,256,0,...,True,False,False,True,False,False,True,False,False,False
3,15.4,135195.3360,16,1.83,0,1,2.7,0,512,0,...,False,True,True,False,False,False,False,False,False,True
4,13.3,96095.8080,8,1.37,0,1,3.1,0,256,0,...,True,False,False,True,False,False,False,False,False,True
